# 02 - Sentiment Modeling Baseline (VADER)

Muc tieu: chay baseline VADER tren processed text de lay lower bound cho bai toan sentiment 3 lop (Negative / Neutral / Positive).

Dau ra chinh: Accuracy, F1-macro, bao cao theo tung lop, confusion matrix, va file tong hop cho luan van Chuong 3.

In [ ]:
from __future__ import annotations

import csv
import importlib
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

def ensure_package(module_name: str, pip_name: str | None = None):
    pip_name = pip_name or module_name
    try:
        return importlib.import_module(module_name)
    except Exception:
        print(f'Installing {pip_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        return importlib.import_module(module_name)

nltk = ensure_package('nltk')
nltk.download('vader_lexicon', quiet=True)

tqdm_module = None
try:
    tqdm_module = importlib.import_module('tqdm')
except Exception:
    tqdm_module = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TEST_FILE = PROCESSED_DIR / 'test.csv'
if not TEST_FILE.exists():
    raise FileNotFoundError(f'Missing split file: {TEST_FILE}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TEST_FILE:', TEST_FILE)

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

LABELS = ['Negative', 'Neutral', 'Positive']
THRESH_POS = 0.05
THRESH_NEG = -0.05
SAMPLE_SIZE = None  # Dat so nguyen (vd 10000) neu muon chay nhanh de debug

def vader_to_label(compound: float) -> str:
    if compound >= THRESH_POS:
        return 'Positive'
    if compound <= THRESH_NEG:
        return 'Negative'
    return 'Neutral'

def compute_classification_report(y_true, y_pred, labels):
    metrics = {}
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
        support = sum(1 for t in y_true if t == label)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

        metrics[label] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': support,
        }

    accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true) if y_true else 0.0
    macro_f1 = sum(metrics[label]['f1'] for label in labels) / len(labels) if labels else 0.0

    confusion = {true_label: {pred_label: 0 for pred_label in labels} for true_label in labels}
    for t, p in zip(y_true, y_pred):
        confusion[t][p] += 1

    return {
        'accuracy': accuracy,
        'f1_macro': macro_f1,
        'per_class': metrics,
        'confusion_matrix': confusion,
    }

In [ ]:
sia = SentimentIntensityAnalyzer()

y_true = []
y_pred = []
compound_scores = []

total_rows = 0
with open(TEST_FILE, 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    rows = reader
    if tqdm_module is not None:
        rows = tqdm_module.tqdm(rows, desc='Running VADER baseline')

    for row in rows:
        total_rows += 1
        if SAMPLE_SIZE is not None and total_rows > SAMPLE_SIZE:
            break

        text = (row.get('text') or '').strip()
        gold = (row.get('sentiment') or '').strip()
        if not text or gold not in LABELS:
            continue

        score = sia.polarity_scores(text)['compound']
        pred = vader_to_label(score)

        y_true.append(gold)
        y_pred.append(pred)
        compound_scores.append(score)

metrics = compute_classification_report(y_true, y_pred, LABELS)

print('Total evaluated samples:', len(y_true))
print('Accuracy:', round(metrics['accuracy'], 6))
print('F1-macro:', round(metrics['f1_macro'], 6))
print('Gold label distribution:', dict(Counter(y_true)))
print('Pred label distribution:', dict(Counter(y_pred)))

In [ ]:
# Hien thi bao cao chi tiet
for label in LABELS:
    m = metrics['per_class'][label]
    print(
        f"{label:8s} | precision={m['precision']:.4f} | recall={m['recall']:.4f} | f1={m['f1']:.4f} | support={m['support']}"
    )

print('\nConfusion matrix (rows=true, cols=pred):')
header = 'true\pred'.ljust(12) + ''.join(lbl.ljust(12) for lbl in LABELS)
print(header)
for true_lbl in LABELS:
    row_txt = true_lbl.ljust(12)
    for pred_lbl in LABELS:
        row_txt += str(metrics['confusion_matrix'][true_lbl][pred_lbl]).ljust(12)
    print(row_txt)

In [ ]:
# Luu metric va doan van lower bound cho luan van Chuong 3
metrics_out = PROCESSED_DIR / 'vader_baseline_metrics.json'
chapter3_out = PROCESSED_DIR / 'chapter3_vader_lower_bound.md'

payload = {
    'model': 'VADER baseline',
    'dataset': str(TEST_FILE),
    'samples': len(y_true),
    'thresholds': {'neg': THRESH_NEG, 'pos': THRESH_POS},
    'accuracy': metrics['accuracy'],
    'f1_macro': metrics['f1_macro'],
    'per_class': metrics['per_class'],
    'confusion_matrix': metrics['confusion_matrix'],
}

with open(metrics_out, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

chapter3_text = f'''### Baseline VADER (Lower Bound)
De xac lap nguong lower bound cho bai toan phan loai sentiment 3 lop, mo hinh luat VADER duoc ap dung truc tiep tren tap test da tien xu ly.
Ket qua thu duoc: Accuracy = {metrics['accuracy']:.4f}, F1-macro = {metrics['f1_macro']:.4f}.
Gia tri nay duoc dung lam moc tham chieu toi thieu de so sanh voi cac mo hinh hoc may va hoc sau trong cac phan tiep theo.
'''

with open(chapter3_out, 'w', encoding='utf-8') as f:
    f.write(chapter3_text)

print('Saved metrics to:', metrics_out)
print('Saved thesis snippet to:', chapter3_out)
print('\n' + chapter3_text)